# 03 — A warmer collective climate accompanies the optimistic shift

Group-level affective tone vs belief change (related, unrelated, content-specificity), decorrelation from baseline, discussion amounts, and the tautology (amount) control. Tone permutations use 10,000 resamples.

**Expected:** tone→related r=.59, permutation p=.005 (null ≈0, SD .25); tone→unrelated r=.18, p=.42; content-specificity Δr=.41, p=.069; decorrelation (tone vs group baseline) r=−.23, p=.38. Related domain discussed M=16% (SD 7%, range 6–31%); unrelated <0.2%. Amount⊥tone r=−.07; amount→shift β=+1.95, p=.43; tone partial r=.60, p=.010. Baseline-dispersion control: tone~dispersion r=−.33 p=.20; dispersion→shift b=−0.69 p=.79; tone partial r=.56 p=.021; tone | dispersion+amount b=+6.54 p=.027.

In [1]:
import pandas as pd, numpy as np, warnings
from scipy import stats
import statsmodels.formula.api as smf
warnings.filterwarnings('ignore')
DATA = '../data'
def load_belief(): return pd.read_excel(f'{DATA}/belief_items.xlsx')
def load_subject(): return pd.read_excel(f'{DATA}/subject_measures.xlsx')
def load_group(): return pd.read_csv(f'{DATA}/group_conversation.csv')
def rep(m,t):
    ci=m.conf_int().loc[t]
    return f'b={m.params[t]:+.3f} 95%CI[{ci[0]:+.3f}, {ci[1]:+.3f}] p={m.pvalues[t]:.4f}'
def corr_ci(x,y):
    r,p=stats.pearsonr(x,y); n=len(x); z=np.arctanh(r); se=1/np.sqrt(n-3)
    return r,np.tanh(z-1.96*se),np.tanh(z+1.96*se),p,n
SEED=20260725; NP=10000


In [2]:
g=load_group(); b=load_belief().dropna(subset=['pre','post'])
def gch(topic):
    e=b[(b.condition=='collaborative')&(b.topic==topic)].copy(); e['ch']=e.post-e.pre
    return e.groupby('group')['ch'].mean()
base=b[(b.condition=='collaborative')&(b.topic=='related')].groupby('group')['pre'].mean()
G=g.set_index('group').join(gch('related').rename('rel')).join(gch('unrelated').rename('unr')).join(base.rename('baseline')).dropna(subset=['rel','tone'])
print('groups:',len(G))
rng=np.random.default_rng(SEED); tv=G.tone.values
def perm_r(y,twotailed=False):
    obs=np.corrcoef(tv,y)[0,1]; nul=np.array([np.corrcoef(rng.permutation(tv),y)[0,1] for _ in range(NP)])
    p=(np.sum(np.abs(nul)>=abs(obs))+1)/(NP+1) if twotailed else (np.sum(nul>=obs)+1)/(NP+1)
    return obs,nul.mean(),nul.std(),p
o,m,s,p=perm_r(G.rel.values); print(f'tone->related  r={o:+.2f} permutation p={p:.3f} (null {m:+.2f}, SD {s:.2f})')
o,m,s,p=perm_r(G.unr.values,twotailed=True); print(f'tone->unrelated r={o:+.2f} permutation p={p:.3f}')
dobs=np.corrcoef(tv,G.rel)[0,1]-np.corrcoef(tv,G.unr)[0,1]
dn=np.array([(lambda pp:np.corrcoef(pp,G.rel)[0,1]-np.corrcoef(pp,G.unr)[0,1])(rng.permutation(tv)) for _ in range(NP)])
print(f'content-specificity Delta_r={dobs:+.2f} permutation p={(np.sum(dn>=dobs)+1)/(NP+1):.3f} (null {dn.mean():+.2f}, SD {dn.std():.2f})')
r,lo,hi,p,n=corr_ci(G.tone,G.baseline); print(f'decorrelation tone~baseline r={r:+.2f} 95%CI[{lo:+.2f},{hi:+.2f}] p={p:.2f} n={n}')


groups: 17
tone->related  r=+0.59 permutation p=0.004 (null -0.00, SD 0.25)
tone->unrelated r=+0.18 permutation p=0.431
content-specificity Delta_r=+0.41 permutation p=0.066 (null +0.00, SD 0.31)
decorrelation tone~baseline r=-0.23 95%CI[-0.64,+0.28] p=0.38 n=17


In [3]:
# discussion amounts (from the transcript-derived group_conversation file; per-concept breakdown needs the transcripts)
print(f'related domain discussed: M={g.related_amount.mean()*100:.0f}% SD={g.related_amount.std()*100:.0f}% range {g.related_amount.min()*100:.0f}-{g.related_amount.max()*100:.0f}%')
print(f'unrelated domain: M={g.unrelated_amount.mean()*100:.2f}%')
# tautology / amount control (G already carries related_amount from the group file)
G2=G.copy()
for c in ['tone','related_amount']: G2[c+'_z']=(G2[c]-G2[c].mean())/G2[c].std()
r_ta=stats.pearsonr(G2.tone, G2.related_amount); ci_ta=r_ta.confidence_interval(0.95)
print(f'corr(tone, amount) = {r_ta.statistic:+.2f} 95%CI[{ci_ta.low:+.2f}, {ci_ta.high:+.2f}] p={r_ta.pvalue:.2f}')
m=smf.ols('rel ~ tone_z + related_amount_z', G2).fit()
print('  amount:', rep(m,'related_amount_z'))
pt=stats.pearsonr(smf.ols('tone_z ~ related_amount_z',G2).fit().resid, smf.ols('rel ~ related_amount_z',G2).fit().resid)
print(f'  tone partial r (| amount) = {pt[0]:+.2f}, p={pt[1]:.3f}')


related domain discussed: M=16% SD=7% range 6-31%
unrelated domain: M=0.01%
corr(tone, amount) = -0.07 95%CI[-0.53, +0.43] p=0.79
  amount: b=+1.946 95%CI[-3.225, +7.117] p=0.4331
  tone partial r (| amount) = +0.61, p=0.010


### Building the discussion amounts from the transcripts (SI Note S7)

`related_amount` and `unrelated_amount` in `data/group_conversation.csv` were computed by the cell below. It matches each transcribed utterance (case-insensitive) against the keyword stems listed in SI Note S7; an utterance touches a domain if it contains any stem of that domain, and each group's amount is the proportion of its utterances that do. **The transcripts are not shared** (participant privacy), so the cell only runs where `TRANSCRIPTS` exists; the output shown was produced on the original transcripts and reproduces `group_conversation.csv` exactly (all 17 groups, differences < 1e-15). Only proportions are printed, never text.

In [4]:
import os, re
TRANSCRIPTS = '../transcripts/utterances.csv'   # NOT SHARED: one row per utterance, columns `group`, `text`
REL = {'students / learning':          r'estudiant|estudiante|alumn|aprend|estudia',
       'system / AI / technology':     r'sistema|intel.lig|inteligenc|\bia\b|\bai\b|tecnolog|algoritm|program|aplicaci',
       'exams / evaluation':           r'examen|avalua|evalua|nota|qualific|calific|prova',
       'stress / anxiety':             r'estr[eé]s|ansi',
       'confidence':                   r'confian',
       'professional / work':          r'profession|profesion|feina|treball|carrera',
       'teacher / school / university':r'professor|profesor|escol|univ|classe|clase'}
UNR = {'lottery / prize':              r'loter|premi|premio|sorteo',
       'first class / travel':         r'primera cl|business class|upgrade',
       'famous / celebrity':           r'famos|famós|celebr',
       'TV / cinema':                  r'concurs|concurso|televisi|estrena|pel.l[ií]cul|pelicul|cine'}
if not os.path.exists(TRANSCRIPTS):
    print('Transcripts not shared; the amounts are read from data/group_conversation.csv (see output below).')
else:
    u = pd.read_csv(TRANSCRIPTS); txt = u.text.astype(str).str.lower()
    for k, p in {**REL, **UNR}.items(): u[k] = txt.str.contains(p, regex=True)
    u['related_amount'] = u[list(REL)].any(axis=1); u['unrelated_amount'] = u[list(UNR)].any(axis=1)
    per_group = u.groupby('group')
    amt = per_group[['related_amount', 'unrelated_amount']].mean()
    print(f'groups: {len(amt)} | utterances: {len(u)}')
    print(f'related domain, across groups: M={amt.related_amount.mean()*100:.1f}% SD={amt.related_amount.std()*100:.1f}% '
          f'range {amt.related_amount.min()*100:.1f}-{amt.related_amount.max()*100:.1f}%')
    print(f'unrelated domain, across groups: M={amt.unrelated_amount.mean()*100:.2f}% max={amt.unrelated_amount.max()*100:.2f}%')
    print('per concept, % of a group\'s utterances (mean and SD across groups):')
    for k in list(REL) + list(UNR): print(f'  {k:30s} M={per_group[k].mean().mean()*100:5.2f}%  SD={per_group[k].mean().std()*100:5.2f}%')

groups: 17 | utterances: 5190
related domain, across groups: M=16.1% SD=6.8% range 6.5-31.1%
unrelated domain, across groups: M=0.01% max=0.17%
per concept, % of a group's utterances (mean and SD across groups):
  students / learning            M= 3.31%  SD= 1.79%
  system / AI / technology       M= 8.78%  SD= 3.26%
  exams / evaluation             M= 2.34%  SD= 2.49%
  stress / anxiety               M= 0.49%  SD= 0.65%
  confidence                     M= 0.14%  SD= 0.29%
  professional / work            M= 1.53%  SD= 1.69%
  teacher / school / university  M= 3.25%  SD= 3.09%
  lottery / prize                M= 0.00%  SD= 0.00%
  first class / travel           M= 0.00%  SD= 0.00%
  famous / celebrity             M= 0.00%  SD= 0.00%
  TV / cinema                    M= 0.01%  SD= 0.04%


In [5]:
# baseline-dispersion control (Results: "The same held for how similar group members' beliefs were to begin with")
b=load_belief().dropna(subset=['pre','post']); g=load_group()
def dispersion(topic, col):
    e=b[(b.condition=='collaborative')&(b.topic==topic)]; piv=e.pivot_table(col,'Item','participant'); out={}
    for grp,mem in e.groupby('group')['participant'].unique().items():
        m=[x for x in mem if x in piv.columns]
        ds=[np.nanmean(np.abs(piv[m[i]]-piv[m[j]])) for i in range(len(m)) for j in range(i+1,len(m))]
        out[grp]=np.mean(ds) if ds else np.nan
    return pd.Series(out)
e=b[(b.condition=='collaborative')&(b.topic=='related')].copy(); e['ch']=e.post-e.pre
G=(g.set_index('group').join(e.groupby('group')['ch'].mean().rename('rel'))
     .join(dispersion('related','pre').rename('disp')).dropna(subset=['rel','tone','disp']))
print(f'groups: {len(G)} | baseline dispersion M={G.disp.mean():.2f} SD={G.disp.std():.2f}')
r,lo,hi,p,n=corr_ci(G.tone,G.disp);  print(f'  tone ~ baseline dispersion  r={r:+.2f} 95%CI[{lo:+.2f}, {hi:+.2f}] p={p:.2f}')
r,lo,hi,p,n=corr_ci(G.disp,G.rel);   print(f'  dispersion ~ related shift  r={r:+.2f} 95%CI[{lo:+.2f}, {hi:+.2f}] p={p:.2f}')
G2=G.copy()
for c in ['tone','disp','related_amount']: G2[c+'_z']=(G2[c]-G2[c].mean())/G2[c].std()
m=smf.ols('rel ~ tone_z + disp_z', G2).fit(); print('  dispersion (with tone):', rep(m,'disp_z'))
pt=stats.pearsonr(smf.ols('tone_z ~ disp_z',G2).fit().resid, smf.ols('rel ~ disp_z',G2).fit().resid)
print(f'  tone partial r (| dispersion) = {pt[0]:+.2f}, p={pt[1]:.3f}')
m3=smf.ols('rel ~ tone_z + disp_z + related_amount_z', G2).fit()
print('  tone (| dispersion + amount):', rep(m3,'tone_z'))

groups: 17 | baseline dispersion M=27.45 SD=7.62
  tone ~ baseline dispersion  r=-0.33 95%CI[-0.70, +0.18] p=0.20
  dispersion ~ related shift  r=-0.25 95%CI[-0.65, +0.26] p=0.34
  dispersion (with tone): b=-0.694 95%CI[-6.270, +4.882] p=0.7933
  tone partial r (| dispersion) = +0.56, p=0.021
  tone (| dispersion + amount): b=+6.536 95%CI[+0.855, +12.218] p=0.0273
